# 🍜 STEP 07 — 오케스트레이터 + 사용자 입력 테스트

## 구조
```
사용자 자연어 입력
    ↓
[OrchestratorAgent]  ← LLM이 도메인 판단
    ├─ "map"      → LocationAgent (상권분석 + MCP)
    ├─ "finance"  → FinanceAgent (BEP 계산)
    ├─ "pipeline" → Step06 파이프라인 (상권→재무→리포트)
    └─ "unknown"  → 안내 메시지
```

## 실행 방법
- **셀 순서대로 실행** 후 마지막 셀에서 테스트
- 시나리오 모드: 미리 준비된 3개 케이스 자동 실행
- 자유입력 모드: `input()`으로 직접 타이핑


In [ ]:
import asyncio, os, json, re
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion, AzureChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.functions import KernelArguments, kernel_function
from semantic_kernel.connectors.mcp import MCPSsePlugin

from semantic_kernel.processes import ProcessBuilder
from semantic_kernel.processes.kernel_process import (
    KernelProcessStep, KernelProcessStepContext,
)
from semantic_kernel.processes.kernel_process.kernel_process_event import KernelProcessEvent
from semantic_kernel.processes.local_runtime.local_kernel_process import start

load_dotenv()

def make_kernel():
    k = Kernel()
    k.add_service(AzureChatCompletion(
        deployment_name=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME'),
        endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
        api_key=os.getenv('AZURE_OPENAI_API_KEY'),
        api_version='2024-08-01-preview',
    ))
    return k

def make_settings(auto=True):
    s = AzureChatPromptExecutionSettings()
    if auto:
        s.function_choice_behavior = FunctionChoiceBehavior.Auto()
    return s

print('✅ 임포트 완료')


## 1. OrchestratorAgent — 도메인 분류
LLM이 사용자 입력을 읽고 `map / finance / pipeline / unknown` 중 하나로 판단합니다.

In [ ]:
async def classify_domain(user_input: str) -> dict:
    """
    OrchestratorAgent가 사용자 입력을 분석하여 도메인과 파라미터를 추출.
    반환: {domain, location, business_type, quarter, raw_input}
    """
    kernel = make_kernel()
    agent = ChatCompletionAgent(
        name='OrchestratorAgent',
        instructions=(
            'You are an orchestrator for an F&B startup support system. '
            'Analyze the user request and return ONLY a JSON object with these fields: '
            '- domain: one of [map, finance, pipeline, unknown] '
            '  map      = asking about commercial area, location, competitor stores '
            '  finance  = asking about BEP, investment, profit, cost calculation '
            '  pipeline = asking for both location AND financial analysis together '
            '  unknown  = out of scope '
            '- location: Korean area name if mentioned (e.g. 홍대, 강남), else null '
            '- business_type: Korean business type if mentioned (e.g. 카페, 한식), else null '
            '- quarter: quarter code YYYYQ if mentioned, else "20253" '
            '- reason: one sentence explaining your domain choice in Korean '
            'Return ONLY the JSON, no markdown fences, no extra text.'
        ),
        kernel=kernel,
        arguments=KernelArguments(settings=make_settings(auto=False)),
    )

    thread = ChatHistoryAgentThread()
    result_text = ''
    async for msg in agent.invoke(messages=user_input, thread=thread):
        result_text += str(msg.content)

    try:
        parsed = json.loads(result_text.strip())
    except:
        # JSON 파싱 실패 시 unknown 처리
        parsed = {'domain': 'unknown', 'location': None, 'business_type': None,
                  'quarter': '20253', 'reason': '입력 파싱 실패'}

    parsed['raw_input'] = user_input
    return parsed

print('✅ OrchestratorAgent 분류 함수 정의 완료')


## 2. 도메인별 핸들러

In [ ]:
# ── map 핸들러: LocationAgent + MCP ──
async def handle_map(params: dict) -> str:
    location      = params.get('location') or '홍대'
    business_type = params.get('business_type') or '카페'
    quarter       = params.get('quarter') or '20253'

    print(f'  [map] {location} / {business_type} / {quarter}')

    kernel = make_kernel()
    mcp_plugin = MCPSsePlugin(
        name='SeoulCommercial',
        description='Seoul commercial area API',
        url='http://localhost:8001/sse',
    )
    await mcp_plugin.connect()
    kernel.add_plugin(mcp_plugin)

    agent = ChatCompletionAgent(
        name='LocationAgent',
        instructions=(
            'You are a commercial area analysis expert for F&B startups in Seoul. '
            'Use get_estimated_sales and get_store_count tools to fetch data, '
            'then synthesize into Risk/Opportunity analysis. '
            'Start with 📅 데이터 기준: YYYY년 N분기. '
            'Always respond in Korean.'
        ),
        kernel=kernel,
        arguments=KernelArguments(settings=make_settings(auto=True)),
    )

    thread = ChatHistoryAgentThread()
    prompt = (
        f'Analyze commercial area for {location}, business type {business_type}, '
        f'quarter {quarter}. Respond in Korean.'
    )
    result = ''
    async for msg in agent.invoke(messages=prompt, thread=thread):
        result += str(msg.content)

    await mcp_plugin.close()
    return result


# ── finance 핸들러: FinanceAgent ──
async def handle_finance(params: dict) -> str:
    location      = params.get('location') or '미지정'
    business_type = params.get('business_type') or '카페'
    raw_input     = params.get('raw_input', '')

    print(f'  [finance] {location} / {business_type}')

    kernel = make_kernel()
    agent = ChatCompletionAgent(
        name='FinanceAgent',
        instructions=(
            'You are a financial analyst for F&B startups in Korea. '
            'Calculate BEP, payback period, and 3 scenarios (pessimistic/base/optimistic). '
            'If specific numbers are not given, use typical Korean cafe/restaurant benchmarks: '
            'investment 80M KRW, monthly fixed cost 4M KRW, variable cost ratio 35%. '
            'Always respond in Korean with specific numbers.'
        ),
        kernel=kernel,
        arguments=KernelArguments(settings=make_settings(auto=False)),
    )

    thread = ChatHistoryAgentThread()
    prompt = (
        f'User request (translate to analysis task): {raw_input}\n'
        f'Location: {location}, Business type: {business_type}. Respond in Korean.'
    )
    result = ''
    async for msg in agent.invoke(messages=prompt, thread=thread):
        result += str(msg.content)
    return result


# ── pipeline 핸들러: Step06 파이프라인 import ──
async def handle_pipeline(params: dict) -> str:
    location      = params.get('location') or '홍대'
    business_type = params.get('business_type') or '카페'
    quarter       = params.get('quarter') or '20253'

    print(f'  [pipeline] {location} / {business_type} / {quarter} → Step06 파이프라인 실행')
    # Step06의 클래스/이벤트는 이 노트북에 직접 인라인
    # (별도 import 없이 아래 셀에서 정의 후 사용)
    return await run_pipeline(location, business_type, quarter, params.get('raw_input',''))


# ── unknown 핸들러 ──
async def handle_unknown(params: dict) -> str:
    return (
        '죄송합니다. 현재 지원하는 기능은 다음과 같습니다:\n'
        '• 상권 분석: "홍대 카페 상권 분석해줘"\n'
        '• 재무 분석: "강남 한식당 손익분기점 계산해줘"\n'
        '• 종합 분석: "잠실 치킨 창업 전체 분석해줘"\n'
    )

print('✅ 도메인 핸들러 정의 완료')


## 3. Step06 파이프라인 (인라인)

In [ ]:
from enum import Enum

class PipelineEvent(str, Enum):
    Start              = "Start"
    LocationDone       = "LocationDone"
    FinanceDoneNeedAlt = "FinanceDoneNeedAlt"
    FinanceDoneOk      = "FinanceDoneOk"
    AltLocationDone    = "AltLocationDone"


class LocationStep(KernelProcessStep):
    @kernel_function(name='analyze_location')
    async def analyze_location(self, kernel: Kernel, request_data: dict,
                                step_context: KernelProcessStepContext) -> None:
        location      = request_data.get('location', '홍대')
        business_type = request_data.get('business_type', '카페')
        quarter       = request_data.get('quarter', '20253')

        print(f'  [Step1] 상권분석: {location}/{business_type}/{quarter}')

        mcp = MCPSsePlugin(name='SC1', description='Seoul API', url='http://localhost:8001/sse')
        await mcp.connect()
        kernel.add_plugin(mcp)

        sales_raw = json.loads(str(await kernel.invoke(
            plugin_name='SC1', function_name='get_estimated_sales',
            location=location, business_type=business_type, quarter=quarter)))
        store_raw = json.loads(str(await kernel.invoke(
            plugin_name='SC1', function_name='get_store_count',
            location=location, business_type=business_type, quarter=quarter)))

        monthly_sales = sales_raw.get('monthly_sales_krw', 0)
        store_count   = store_raw.get('store_count', 1)
        print(f'  [Step1] 월매출 {monthly_sales:,}원 / 점포 {store_count}개 → Step2 전달')

        await step_context.emit_event(
            process_event=PipelineEvent.LocationDone,
            data={**request_data, 'sales_raw': sales_raw, 'store_raw': store_raw,
                  'location_analysis': f'{location} {business_type} 상권: 월매출 {monthly_sales:,}원, 점포수 {store_count}개'},
        )


class FinanceStep(KernelProcessStep):
    @kernel_function(name='analyze_finance')
    async def analyze_finance(self, kernel: Kernel, location_payload: dict,
                               step_context: KernelProcessStepContext) -> None:
        sales_raw     = location_payload['sales_raw']
        store_raw     = location_payload['store_raw']
        monthly_sales = sales_raw.get('monthly_sales_krw', 0)
        store_count   = store_raw.get('store_count', 1)
        per_store     = monthly_sales // max(store_count, 1)

        print(f'  [Step2] 재무분석: 점포당 월매출 {per_store:,}원')

        # 간단한 BEP 계산 (LLM 없이 수식으로)
        fixed_cost    = 4_000_000
        variable_rate = 0.35
        bep           = int(fixed_cost / (1 - variable_rate))  # ~6,153,846원
        payback       = int(80_000_000 / max(per_store - bep, 1))
        feasibility   = '어려움' if payback > 24 else ('보통' if payback > 12 else '양호')

        print(f'  [Step2] BEP {bep:,}원 / 투자회수 {payback}개월 / {feasibility} → {"Step3" if feasibility=="어려움" else "Step4"}')

        payload = {**location_payload,
                   'bep_monthly_sales': bep, 'payback_months': payback,
                   'feasibility': feasibility, 'per_store_sales': per_store}

        event = PipelineEvent.FinanceDoneNeedAlt if feasibility == '어려움' else PipelineEvent.FinanceDoneOk
        await step_context.emit_event(process_event=event, data=payload)


class AltLocationStep(KernelProcessStep):
    NEARBY: dict = {'홍대': ['합정','연남동'], '강남': ['역삼','선릉'],
                    '이태원': ['한남동'], '건대': ['성수동'], '잠실': ['송파']}

    @kernel_function(name='find_alt_location')
    async def find_alt_location(self, kernel: Kernel, finance_payload: dict,
                                 step_context: KernelProcessStepContext) -> None:
        location      = finance_payload['location']
        business_type = finance_payload['business_type']
        quarter       = finance_payload['quarter']
        bep           = finance_payload.get('bep_monthly_sales', 0)
        candidates    = self.NEARBY.get(location, ['합정'])

        print(f'  [Step3] 대안 조회: {candidates}')

        mcp = MCPSsePlugin(name='SC3', description='Seoul API', url='http://localhost:8001/sse')
        await mcp.connect()
        kernel.add_plugin(mcp)

        alt_results = []
        for c in candidates:
            try:
                s = json.loads(str(await kernel.invoke(
                    plugin_name='SC3', function_name='get_estimated_sales',
                    location=c, business_type=business_type, quarter=quarter)))
                t = json.loads(str(await kernel.invoke(
                    plugin_name='SC3', function_name='get_store_count',
                    location=c, business_type=business_type, quarter=quarter)))
                per = s.get('monthly_sales_krw',0) // max(t.get('store_count',1),1)
                alt_results.append({'location': c, 'per_store_sales': per,
                                    'store_count': t.get('store_count',0)})
                print(f'  [Step3] {c}: 점포당 {per:,}원')
            except Exception as e:
                print(f'  [Step3] {c} 조회 실패: {e}')

        await step_context.emit_event(
            process_event=PipelineEvent.AltLocationDone,
            data={**finance_payload, 'alt_locations': alt_results})


class ReportStep(KernelProcessStep):
    @kernel_function(name='generate_report')
    async def generate_report(self, kernel: Kernel, full_payload: dict,
                               step_context: KernelProcessStepContext) -> None:
        location      = full_payload['location']
        business_type = full_payload['business_type']
        quarter       = full_payload['quarter']
        per_store     = full_payload.get('per_store_sales', 0)
        bep           = full_payload.get('bep_monthly_sales', 0)
        payback       = full_payload.get('payback_months', 0)
        feasibility   = full_payload.get('feasibility', '보통')
        alt_locations = full_payload.get('alt_locations', [])

        print(f'  [Step4] 최종 리포트 생성')

        alt_text = ''
        if alt_locations:
            alt_text = '\n\n[대안 지역 비교]\n'
            for a in alt_locations:
                flag = '✅' if a['per_store_sales'] >= bep else '❌'
                alt_text += f"  {flag} {a['location']}: 점포당 {a['per_store_sales']:,}원 (점포수 {a['store_count']}개)\n"

        report = (
            f'\n{"="*55}\n'
            f'📋 {location} {business_type} 창업 타당성 분석\n'
            f'{"="*55}\n'
            f'📅 데이터 기준: {quarter[:4]}년 {quarter[4]}분기\n\n'
            f'[상권 현황]\n'
            f'  점포당 예상 월매출: {per_store:,}원\n'
            f'  경쟁 점포수: {full_payload["store_raw"].get("store_count",0)}개\n\n'
            f'[재무 타당성]\n'
            f'  손익분기 월매출: {bep:,}원\n'
            f'  투자 회수 예상: {payback}개월\n'
            f'  종합 판정: {feasibility}\n'
            f'{alt_text}'
            f'{"="*55}'
        )
        print(report)
        full_payload['final_report'] = report


async def run_pipeline(location, business_type, quarter, raw_input=''):
    builder = ProcessBuilder(name='OrchestratorPipeline')
    s1 = builder.add_step(LocationStep)
    s2 = builder.add_step(FinanceStep)
    s3 = builder.add_step(AltLocationStep)
    s4 = builder.add_step(ReportStep)

    builder.on_input_event(PipelineEvent.Start) \
        .send_event_to(s1, function_name='analyze_location', parameter_name='request_data')
    s1.on_event(PipelineEvent.LocationDone) \
        .send_event_to(s2, function_name='analyze_finance', parameter_name='location_payload')
    s2.on_event(PipelineEvent.FinanceDoneNeedAlt) \
        .send_event_to(s3, function_name='find_alt_location', parameter_name='finance_payload')
    s2.on_event(PipelineEvent.FinanceDoneOk) \
        .send_event_to(s4, function_name='generate_report', parameter_name='full_payload')
    s3.on_event(PipelineEvent.AltLocationDone) \
        .send_event_to(s4, function_name='generate_report', parameter_name='full_payload')

    pipeline = builder.build()
    await start(
        process=pipeline,
        kernel=make_kernel(),
        initial_event=KernelProcessEvent(
            id=PipelineEvent.Start,
            data={'request': raw_input, 'location': location,
                  'business_type': business_type, 'quarter': quarter},
        ),
    )
    return '파이프라인 실행 완료 (위 출력 참조)'

print('✅ 파이프라인 정의 완료')


## 4. 오케스트레이터 메인 함수

In [ ]:
async def orchestrate(user_input: str):
    """사용자 입력 → 도메인 분류 → 에이전트 라우팅 → 결과 반환"""
    print()
    print('─' * 55)
    print(f'👤 사용자: {user_input}')
    print('─' * 55)

    # 1. 도메인 분류
    params = await classify_domain(user_input)
    domain = params.get('domain', 'unknown')
    print(f'🧭 오케스트레이터 판단: [{domain}] — {params.get("reason","")}')
    print(f'   파라미터: location={params.get("location")}, '
          f'business_type={params.get("business_type")}, quarter={params.get("quarter")}')
    print()

    # 2. 라우팅
    handlers = {
        'map':      handle_map,
        'finance':  handle_finance,
        'pipeline': handle_pipeline,
        'unknown':  handle_unknown,
    }
    handler = handlers.get(domain, handle_unknown)
    result  = await handler(params)

    # 3. 결과 출력
    print()
    print('🤖 에이전트 응답:')
    print(result)
    print('─' * 55)
    return result

print('✅ 오케스트레이터 메인 함수 정의 완료')


## 5. 시나리오 테스트
미리 준비된 3개 케이스를 순서대로 실행합니다.

⚠️ MCP 서버(`seoul_commercial_mcp_server_sse.py`) 실행 중이어야 합니다.

In [ ]:
SCENARIOS = [
    # 케이스 1: 상권 분석만
    "홍대에서 카페 창업하려는데 상권 분석해줘",

    # 케이스 2: 재무 분석만
    "강남에 한식당 열면 손익분기점이 얼마야?",

    # 케이스 3: 종합 분석 (파이프라인)
    "잠실에 치킨집 창업하고 싶어. 상권이랑 수익성 전체 다 분석해줘",
]

for i, scenario in enumerate(SCENARIOS, 1):
    print(f'\n[시나리오 {i}/{len(SCENARIOS)}]')
    await orchestrate(scenario)
    print()


## 6. 자유 입력 모드
직접 입력하며 테스트합니다. `종료` 입력 시 종료.

In [ ]:
print('자유 입력 모드 시작 (종료하려면 "종료" 입력)')
print()

while True:
    try:
        user_input = input('👤 입력: ').strip()
    except EOFError:
        break

    if not user_input:
        continue
    if user_input in ('종료', 'exit', 'quit'):
        print('종료합니다.')
        break

    await orchestrate(user_input)
